In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
import torch
print(torch.cuda.device_count())   

1


In [34]:
from datasets import load_dataset

data = load_dataset("json", data_files="data/my_train.jsonl", split="train")
data = data.train_test_split(test_size=0.1, seed=42)
print(data)

Generating train split: 15606 examples [00:00, 802495.01 examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'score'],
        num_rows: 14045
    })
    test: Dataset({
        features: ['text', 'score'],
        num_rows: 1561
    })
})


## tokenization

In [35]:
from transformers import AutoTokenizer

model_name = "AI-Sweden-Models/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [36]:
enc = tokenizer("Den gamla bron renoverades.")
enc

{'input_ids': [3, 1685, 3796, 22876, 25324, 706, 20, 4], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}

In [37]:
tokenizer.convert_ids_to_tokens(enc["input_ids"])

['[CLS]', 'Den', 'Ġgamla', 'Ġbron', 'Ġrenovera', 'des', '.', '[SEP]']

In [38]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

data = data.map(tokenize, batched= True)

Map: 100%|██████████| 1561/1561 [00:00<00:00, 20329.97 examples/s]


In [39]:
data = data.rename_column("score", "labels")

In [40]:
print(data)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 14045
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1561
    })
})


In [41]:
data = data.remove_columns(["token_type_ids"])

In [42]:
print(data)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 14045
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 1561
    })
})


## loading the model

In [53]:
from transformers import set_seed
set_seed(42)

In [54]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1, reference_compile=False)

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at AI-Sweden-Models/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [55]:
n_params = sum(p.numel() for p in model.parameters())
n_params

149605633

first trying the untrained model

In [56]:
import torch

for sentence in ["Den gamla bron renoverades.", "Den gamla bro renoverades."]:
    enc = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        out = model(input_ids=enc["input_ids"], attention_mask = enc["attention_mask"])
        print(sentence, out.logits.item())


Den gamla bron renoverades. -0.5433064103126526
Den gamla bro renoverades. 0.650090217590332


## now setting up the trainer

In [57]:
from scipy.stats import spearmanr

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.squeeze()
    result = spearmanr(predictions, labels)
    return {"spearmanr": result.correlation}

In [58]:
# settings
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="checkpoints",
    num_train_epochs= 3,
    learning_rate= 2e-5,
    per_device_train_batch_size= 32, 
    eval_strategy= "epoch",
    logging_steps=50,
    seed=42,
    report_to="none",
    warmup_ratio=0.1
)

In [59]:
from transformers import Trainer, DataCollatorWithPadding

trainer = Trainer(
    model=model,
    args= args,
    train_dataset=data["train"],
    eval_dataset=data["test"],
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

In [60]:
trainer.train_dataset = data["train"]
trainer.train()

Epoch,Training Loss,Validation Loss,Spearmanr
1,0.121200,0.084109,0.023505
2,0.101500,0.097221,0.453149
3,0.051600,0.056786,0.503192


TrainOutput(global_step=1317, training_loss=0.16833404310743466, metrics={'train_runtime': 332.6349, 'train_samples_per_second': 126.67, 'train_steps_per_second': 3.959, 'total_flos': 2161407374012886.0, 'train_loss': 0.16833404310743466, 'epoch': 3.0})

In [61]:
# saving the model
trainer.save_model("my_model")
tokenizer.save_pretrained("my_model")

('my_model/tokenizer_config.json',
 'my_model/special_tokens_map.json',
 'my_model/tokenizer.json')

In [62]:
# test
model.eval()
for s in ["Den gamla bron renoverades.", "Den gamla bro renoverades."]:
    inputs = tokenizer(s, return_tensors="pt").to(model.device)
    with torch.no_grad():
        print(s, model(**inputs).logits.item())

Den gamla bron renoverades. 0.9927250146865845
Den gamla bro renoverades. 0.3876308798789978


---

## analysis



| Metric   | Run 1 (DaLAJ + toy) | Run 2 (+ScaLA) | Run 3 (+warmup, lr = 2e-5) |
|-----------|--------------------|----------------|----------------------------|
| Val loss  | 0.0765             | 0.0773         | 0.0568                     |
| Spearman  | 0.05               | -0.06          | 0.50                       |

so it is doing what a lazy model would do so collapsing to the mean

In [ ]:
# check 1 does it give everyone the same score
out = trainer.predict(data["test"])
preds = out.predictions.squeeze()
print(preds.min(), preds.max(), preds.std())

0.36148292 0.92724997 0.061461348


In [51]:
ex = data["train"][0]
print(tokenizer.decode(ex["input_ids"]))
print(ex["labels"])
print(data["train"].column_names)

[CLS]Men sakta, sakta, med min mans hjälp, kunde jag förstå att mitt tidigare hem inte var det som var viktigt, men att det var min familj som var viktig.[SEP]
0.4
['text', 'labels', 'input_ids', 'attention_mask']


In [52]:
for log in trainer.state.log_history[:10]:
    print(log)

{'loss': 1.4289, 'grad_norm': 103.53656768798828, 'learning_rate': 2.8883826879271073e-05, 'epoch': 0.11389521640091116, 'step': 50}
{'loss': 0.4, 'grad_norm': 193.13693237304688, 'learning_rate': 2.7744874715261958e-05, 'epoch': 0.22779043280182232, 'step': 100}
{'loss': 0.2123, 'grad_norm': 44.96234893798828, 'learning_rate': 2.660592255125285e-05, 'epoch': 0.3416856492027335, 'step': 150}
{'loss': 0.1967, 'grad_norm': 32.960575103759766, 'learning_rate': 2.5466970387243738e-05, 'epoch': 0.45558086560364464, 'step': 200}
{'loss': 0.1089, 'grad_norm': 57.393436431884766, 'learning_rate': 2.4328018223234623e-05, 'epoch': 0.5694760820045558, 'step': 250}
{'loss': 0.1105, 'grad_norm': 64.90243530273438, 'learning_rate': 2.3189066059225514e-05, 'epoch': 0.683371298405467, 'step': 300}
{'loss': 0.2154, 'grad_norm': 52.900108337402344, 'learning_rate': 2.2050113895216403e-05, 'epoch': 0.7972665148063781, 'step': 350}
{'loss': 0.2527, 'grad_norm': 329.7322082519531, 'learning_rate': 2.091116